# DATA 612 — Criteo data prep on Colab (replaces the Zaratan run)

Runs the repo's own pipeline (`scripts/prepare_criteo.py`, merged in PR #1) on Colab and saves the
processed tables to Google Drive, so the whole team can train from the same data.

**Runtime:** Runtime → Change runtime type → **CPU** + **High-RAM** (no GPU needed).
The Zaratan job used 48 GB for both window sizes; 12h alone with 16 shards should fit in Colab High-RAM.

**Steps:** setup → download raw file to Drive (once) → tests → smoke run → full 12h run → check against
the PR numbers → copy to Drive. Then share the Drive folder `DATA612_data` with the team.

## 1. Settings

In [ ]:
import os, sys, json, shutil, subprocess, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/DATA612_data")   # shared team folder
    REPO_DIR  = Path("/content/repo")
else:                                                          # local test run
    DRIVE_DIR = Path(os.environ.get("DRIVE_DIR", "drive_sim/DATA612_data")).resolve()
    REPO_DIR  = Path(os.environ.get("REPO_DIR", "repo")).resolve()

REPO_URL   = "https://github.com/Abhyudaya01/criteo-customer-journey-transformers.git"
RAW_NAME   = "criteo_attribution_dataset/criteo_attribution_dataset.tsv.gz"   # matches configs/base.yaml
RUN_SMOKE  = True     # ~2% of users, quick end-to-end check
RUN_12H    = True     # full primary dataset (Tier 1)
RUN_6H     = False    # 6h comparison (Tier 2, 247M positions) — turn on later
N_SHARDS   = 16       # more shards = less RAM per step

(DRIVE_DIR / "raw").mkdir(parents=True, exist_ok=True)
(DRIVE_DIR / "processed").mkdir(parents=True, exist_ok=True)
print("Drive folder:", DRIVE_DIR)

## 2. Get the repo and dependencies

In [ ]:
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=False)
elif IN_COLAB:
    subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print("Repo:", REPO_DIR, "| commit:",
      subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip())

import importlib.util
need = [p for p, mod in {"pyarrow": "pyarrow", "pyyaml": "yaml", "huggingface_hub": "huggingface_hub",
                         "pytest": "pytest", "psutil": "psutil"}.items()
        if importlib.util.find_spec(mod) is None]
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)

import psutil, pandas as pd
ram_gb = psutil.virtual_memory().total / 1e9
print(f"RAM: {ram_gb:.0f} GB | pandas {pd.__version__}")
if RUN_12H and ram_gb < 40:
    print("WARNING: less than 40 GB RAM. Switch to a High-RAM runtime before the full run.")

## 3. Raw file: download to Drive once, then use a fast local copy
If the download fails with 401/403: open https://huggingface.co/datasets/criteo/criteo-attribution-dataset,
accept the terms, and run `from huggingface_hub import notebook_login; notebook_login()` with a read token.

In [ ]:
drive_raw = DRIVE_DIR / "raw" / RAW_NAME
if not drive_raw.exists():
    from huggingface_hub import hf_hub_download
    t0 = time.time()
    hf_hub_download(repo_id="criteo/criteo-attribution-dataset", repo_type="dataset",
                    filename="criteo_attribution_dataset.tsv.gz", local_dir=str(drive_raw.parent))
    print(f"Downloaded in {time.time() - t0:.0f}s")
print(f"Raw file on Drive: {drive_raw} ({drive_raw.stat().st_size / 1e6:.0f} MB)")

# Local working copy (Drive is slow for heavy reads/writes)
local_raw = REPO_DIR / "data" / "raw" / RAW_NAME
local_raw.parent.mkdir(parents=True, exist_ok=True)
if not local_raw.exists() or local_raw.stat().st_size != drive_raw.stat().st_size:
    shutil.copy2(drive_raw, local_raw)
print("Local copy:", local_raw)

## 4. Tests and schema check

In [ ]:
def run(cmd):
    print("$", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-4000:], r.stderr[-2000:] if r.returncode else "")
    if r.returncode:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")

run([sys.executable, "-m", "pytest", "-q"])
run([sys.executable, "scripts/prepare_criteo.py", "--check-only"])

## 5. Smoke run (~2% of users, all 30 days)

In [ ]:
def show_meta(name):
    meta = json.loads((REPO_DIR / "data/processed" / name / "meta.json").read_text())
    keys = ["n_windows", "n_impressions", "n_users", "n_campaigns", "n_conversions_total",
            "n_positions", "n_positive_positions", "positive_rate_conversion",
            "frac_conversion_windows_covered"]
    display(pd.Series({k: meta[k] for k in keys}, name=name).to_frame())
    return meta

if RUN_SMOKE:
    t0 = time.time()
    run([sys.executable, "scripts/prepare_criteo.py", "--user-sample", "50", "--hours", "12",
         "--tag", "smoke", "--n-shards", "4"])
    print(f"Smoke run: {time.time() - t0:.0f}s")
    show_meta("criteo_w12h_smoke")

## 6. Full 12h run (primary dataset)

In [ ]:
if RUN_12H:
    t0 = time.time()
    run([sys.executable, "scripts/prepare_criteo.py", "--hours", "12", "--n-shards", str(N_SHARDS)])
    print(f"Full 12h run: {(time.time() - t0) / 60:.1f} min")
if RUN_6H:
    t0 = time.time()
    run([sys.executable, "scripts/prepare_criteo.py", "--hours", "6", "--n-shards", str(N_SHARDS * 2)])
    print(f"Full 6h run: {(time.time() - t0) / 60:.1f} min")

## 7. Check against the Zaratan numbers (PR #1, 21 Oct)
Every row should say `OK`. A mismatch means something differs from the Zaratan run — stop and check before training.

In [ ]:
EXPECTED = {
    "criteo_w12h": {"n_windows": 62, "n_impressions": 16_468_027, "n_users": 6_142_256,
                    "n_campaigns": 675, "n_conversions_total": 438_730,
                    "n_positions": 125_971_114, "n_positive_positions": 231_004},
    "criteo_w6h":  {"n_windows": 124, "n_positions": 247_454_311, "n_positive_positions": 236_046},
}
rows = []
for name, exp in EXPECTED.items():
    path = REPO_DIR / "data/processed" / name / "meta.json"
    if not path.exists():
        continue
    meta = json.loads(path.read_text())
    for k, v in exp.items():
        rows.append({"dataset": name, "field": k, "expected": v, "colab": meta.get(k),
                     "check": "OK" if meta.get(k) == v else "MISMATCH"})
check = pd.DataFrame(rows)
display(check)
if len(check) and (check["check"] == "MISMATCH").any():
    print("Some values differ from the Zaratan run. Do not train on this data until resolved.")
elif len(check):
    print("Matches the Zaratan run.")

## 8. Copy processed data to Drive

In [ ]:
src = REPO_DIR / "data" / "processed"
for d in sorted(p for p in src.iterdir() if p.is_dir()):
    dst = DRIVE_DIR / "processed" / d.name
    t0 = time.time()
    shutil.copytree(d, dst, dirs_exist_ok=True)
    size = sum(f.stat().st_size for f in dst.rglob("*") if f.is_file()) / 1e9
    print(f"{d.name:<22} -> {dst}  ({size:.2f} GB, {time.time() - t0:.0f}s)")

print("\nShare this folder with the team (right-click in Drive -> Share):", DRIVE_DIR)
print("Read a table later with: pd.read_parquet(DRIVE_DIR / 'processed/criteo_w12h/positions')")